## Imports

In [ ]:
import numpy as np
from keras.models import load_model
from sklearn.metrics import roc_auc_score, roc_curve
from methods.load_data import load_ucsf
from methods.visualisation import plot_pipeline_examples

IMAGE_SIZE = 128
CLASSIFIER_PATH = "best_classification_model.keras"
SEGMENTATION_PATH = "best_segmentation_model.keras"

## Get dataset

In [ ]:
_, _, x_val, y_val, x_test, y_test = load_ucsf(debug=False, classification=True, image_size=IMAGE_SIZE)
_, _, _, _, x_test_seg, y_test_seg = load_ucsf(debug=False, classification=False, basic_mask=True, image_size=IMAGE_SIZE)

y_test = np.asarray(y_test).astype(int)
y_val = np.asarray(y_val).astype(int)
positive = y_test == 1

assert positive.sum() == len(x_test_seg), (positive.sum(), len(x_test_seg)) # Check the sizes are the same
assert np.allclose(x_test[positive][:200], x_test_seg[:200])

y_test_seg = (np.asarray(y_test_seg).reshape(-1, IMAGE_SIZE, IMAGE_SIZE, 1) > 0)
true_masks = np.zeros((len(x_test), IMAGE_SIZE, IMAGE_SIZE, 1), dtype=bool)
true_masks[positive] = y_test_seg
print(f"Test slices: {len(x_test)}, tumour-positive: {positive.sum()}")

## Load models

In [ ]:
classifier = load_model(CLASSIFIER_PATH, compile=False)
segmenter = load_model(SEGMENTATION_PATH, compile=False)

print(classifier.input_shape, segmenter.input_shape)
assert classifier.input_shape[1:3] == (IMAGE_SIZE, IMAGE_SIZE)
assert segmenter.input_shape[1:3] == (IMAGE_SIZE, IMAGE_SIZE)

## Set Threshold for Classifier

In [ ]:
val_proba = classifier.predict(x_val, batch_size=64, verbose=0).ravel()
fpr, tpr, thresholds = roc_curve(y_val, val_proba)    # False and true positive rate
CLASS_THRESHOLD = thresholds[np.argmax(tpr - fpr)]    # Youden's J
SEG_THRESHOLD = 0.5
print(f"Classifier threshold: {CLASS_THRESHOLD:.3f}")

## Run Combined Model Pipeline

In [ ]:
def run_pipeline(x, classifier, segmenter, class_threshold=CLASS_THRESHOLD, seg_threshold=SEG_THRESHOLD, batch_size=64):
    """Classify every slice, then segment only the flagged ones."""
    class_proba = classifier.predict(x, batch_size=batch_size, verbose=0).ravel()
    flagged = class_proba >= class_threshold
    masks = np.zeros((len(x), IMAGE_SIZE, IMAGE_SIZE, 1), dtype=bool)
    if flagged.any():
        seg_proba = segmenter.predict(x[flagged], batch_size=batch_size, verbose=0)
        masks[flagged] = seg_proba >= seg_threshold
    return class_proba, flagged, masks

class_proba, flagged, pred_masks = run_pipeline(x_test, classifier, segmenter)

## Evaluate Classification

In [ ]:
tp = int((flagged & positive).sum())
fn = int((~flagged & positive).sum())
fp = int((flagged & ~positive).sum())
tn = int((~flagged & ~positive).sum())

print(f"Test AUC: {roc_auc_score(y_test, class_proba):.4f}")
print(f"Flagged correctly: {tp}, missed tumour slices: {fn}, false alarms: {fp}, correct rejections: {tn}")
print(f"Sensitivity: {tp / (tp + fn):.3f}, Specificity: {tn / (tn + fp):.3f}")

## Evaluate Segmentation

In [ ]:
def iou_counts(pred, true):
    pred = pred.reshape(len(pred), -1).astype(bool)
    true = true.reshape(len(true), -1).astype(bool)
    return (pred & true).sum(axis=1), (pred | true).sum(axis=1)

oracle_pred = segmenter.predict(x_test_seg, batch_size=64, verbose=0) >= SEG_THRESHOLD
inter, union = iou_counts(oracle_pred, y_test_seg)
print(f"Segmenter alone | pooled IoU: {inter.sum() / union.sum():.4f} | mean per-slice IoU: {np.mean(inter / union):.4f}")

inter, union = iou_counts(pred_masks, true_masks)
print(f"Pipeline (tumour) | mean per-slice IoU: {np.mean(inter[positive] / union[positive]):.4f}")

print(f"Pipeline (all) | pooled IoU: {inter.sum() / union.sum():.4f}")

## Plot Pipeline Segmentation Results

In [ ]:
def picks_labels():
    rng = np.random.default_rng(1606009)
    groups = [("Correct flag", np.where(flagged & positive)[0]),
              ("Missed tumour", np.where(~flagged & positive)[0]),
              ("False alarm", np.where(flagged & ~positive)[0])]
    picks, labels = [], []
    for name, pool in groups:
        if len(pool):
            chosen = rng.choice(pool, size=min(2, len(pool)), replace=False)
            picks += list(chosen)
            labels += [name] * len(chosen)
    return picks, labels

picks, labels = picks_labels()
plot_pipeline_examples(x_test, true_masks, class_proba, pred_masks, picks, labels)